In [2]:
import pandas as pd
import numpy as np

# Carga del dataset (JSON). 
df_raw = pd.read_json('../data/raw/streaming_users_dirty.json')

# Preservar el original inmutable y trabajar sobre una copia
df = df_raw.copy()

print("=== Dataset original cargado ===")
print(df.head())
print(f"\nForma: {df.shape}")

=== Dataset original cargado ===
   user_id  age subscription_plan  monthly_watch_time_mins   country  \
0    10000   39          Estándar                    805.8    Brasil   
1    10001   37          Estándar                   1173.4  Colombia   
2    10002   28            Básico                    401.0  Colombia   
3    10003   43            Básico                     62.4   Uruguay   
4    10004   51            Básico                    477.8      Perú   

  favorite_genre last_login_date  customer_support_tickets  
0          Crime      2025-03-04                        99  
1          Crime      2019-04-02                         2  
2          Crime      2018-04-13                         0  
3       Thriller      2021-01-31                         0  
4       Thriller      2020-09-30                         1  

Forma: (8160, 8)


In [3]:
# Dimensiones
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")

# Tipos de dato y nulos
print(df.info())

# Estadísticas descriptivas de TODO
print(df.describe(include='all'))

Filas: 8160 | Columnas: 8
<class 'pandas.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8160 non-null   int64  
 1   age                       8160 non-null   int64  
 2   subscription_plan         8160 non-null   str    
 3   monthly_watch_time_mins   7967 non-null   float64
 4   country                   8160 non-null   str    
 5   favorite_genre            7920 non-null   str    
 6   last_login_date           7840 non-null   str    
 7   customer_support_tickets  8160 non-null   int64  
dtypes: float64(1), int64(3), str(4)
memory usage: 756.7 KB
None
             user_id          age subscription_plan  monthly_watch_time_mins  \
count    8160.000000  8160.000000              8160              7967.000000   
unique           NaN          NaN                15                      NaN   
top              NaN    

- El dataset tiene 8160 filas y 8 columnas.
- Se puede ver que las columnas "monthly_watch_time_mins", "favorite_genre" y "last_login_date" no tienen la misma cantidad de registros que el total. --> Revisar de que se trata.
- "last_login_date" refiere a una fecha pero es de tipo texto.
- Se ven mínimos negativos en "monthly_watch_time_mins" y "customer_support_tickets". Las mismas columnas tienen máximos enormes en comparación de la distribución de cuartiles.

In [4]:
# Conteo y porcentaje de faltantes por columna
faltantes = df.isnull().sum()
porcentaje = (faltantes / len(df) * 100).round(2)
resumen_faltantes = pd.DataFrame({'Faltantes': faltantes, '%': porcentaje})
print(resumen_faltantes[resumen_faltantes['Faltantes'] > 0])

                         Faltantes     %
monthly_watch_time_mins        193  2.37
favorite_genre                 240  2.94
last_login_date                320  3.92


In [5]:
# Duplicados exactos (toda la fila repetida)
print(f"Filas duplicadas exactas: {df.duplicated().sum()}")

# user_id no debería repetirse
print(f"user_id repetidos: {df['user_id'].duplicated().sum()}")

Filas duplicadas exactas: 126
user_id repetidos: 160


In [6]:
# Revisar valores únicos y frecuencias de cada categórica
categoricas = df.select_dtypes(include='object').columns
for col in categoricas:
    print(f"\n=== {col} ({df[col].nunique(dropna=True)} valores únicos) ===")
    print(df[col].value_counts(dropna=False))


=== subscription_plan (15 valores únicos) ===
subscription_plan
Básico       3450
Estándar     2711
Premium      1519
basico         60
BASICO         52
Basic          52
básico         50
Std            48
Estándar       46
estandar       36
STANDARD       34
Premium        31
PREMIUM        26
Premiun        23
premium        22
Name: count, dtype: int64

=== country (26 valores únicos) ===
country
Brasil        1132
Chile         1132
México        1129
Uruguay       1124
Perú          1120
Colombia      1116
Argentina     1087
colombia        27
méxico          25
uruguay         24
Brazil          21
COL             19
CHL             18
URY             17
MEX             16
Chile           16
argentina       16
PER             16
chile           15
Mexico          15
Peru            15
BRA             15
brasil          13
perú            12
ARG             10
Argentina       10
Name: count, dtype: int64

=== favorite_genre (28 valores únicos) ===
favorite_genre
Comedia        

C:\Users\Usuario\AppData\Local\Temp\ipykernel_22992\1142984975.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categoricas = df.select_dtypes(include='object').columns


Variables categoricas repetidas
* Aparecen escritas de distinta forma, por ejemplo: Estándar, Std, STANDARD, estandar. Básico, basico, BASICO, Basic, etc. Premium, PREMIUM, Premiun, premium. Lo mismo para las demás. 
* Los valores normales aqui serán: Básico, Estandar y Premium.
* En "country" los normales serán: Brasil, México, Chile, Uruguay, Perú, Colombia, Argentina.
* En "favorite_genre" los normales serán: Comedia, Drama, Documental, Thriller, Romance, Acción, Crimen.
* En "last_login_date" primero se convertirá la variable a tipo Date y si me acuerdo como, a formato "AAAA-MM-DD"

In [7]:
# Revisar rangos de numéricas  
numericas = ['age', 'monthly_watch_time_mins', 'customer_support_tickets']
print(df[numericas].describe().round(2))

# Buscar valores imposibles de forma explícita
print("\nEdades fuera de un rango humano razonable:")
print(df[(df['age'] < 0) | (df['age'] > 120)][['user_id', 'age']])

print("\nMinutos vistos negativos o desmesurados:")
print(df[(df['monthly_watch_time_mins'] < 0) | (df['monthly_watch_time_mins'] > 50000)][['user_id', 'monthly_watch_time_mins']])

print("\nTickets de soporte negativos:")
print(df[df['customer_support_tickets'] < 0][['user_id', 'customer_support_tickets']])

           age  monthly_watch_time_mins  customer_support_tickets
count  8160.00                  7967.00                   8160.00
mean     34.10                  1107.35                      1.80
std      14.51                  5310.44                     11.33
min      -5.00                  -120.00                     -1.00
25%      25.00                   489.20                      0.00
50%      33.00                   757.40                      1.00
75%      42.00                  1045.70                      1.00
max     150.00                 99999.00                    150.00

Edades fuera de un rango humano razonable:
      user_id  age
194     10194  130
324     10324  130
426     10426  150
442     10442   -5
529     10529  130
...       ...  ...
7517    17517   -5
7633    17633  130
7748    17748  130
7790    17790   -5
7967    17967   -5

[74 rows x 2 columns]

Minutos vistos negativos o desmesurados:
      user_id  monthly_watch_time_mins
430     10430                 

* Minimos negativos en "age", "monthly_watch_time_mins" y "customer_support_tickets"
* Edades fuera de rango razonable (130, 150, -5)
* Minutos desmesurados (9999, -1, -120)

En resumen:
* Numéricas: "age", "monthly_watch_time_mins", "customer_support_tickets" y "last_login_date".
* Categóricas: "subscription_plan", "countr"y", "favorite_genre".



Preguntas que se pueden analizar

* Sobre el consumo
1. El tiempo de visualización mensual (`monthly_watch_time_mins`) difiere entre los planes de suscripción (`subscription_plan`)?
2. Existe relación entre la edad del usuario (`age`) y los minutos vistos?
3. Los usuarios que hace más días que no inician sesión (`last_login_date`) ven menos contenido?

* Sobre su ubicación geográfica
4. Qué países (`country`) concentran a los usuarios más activos (mayor promedio de minutos vistos)?
5. El plan contratado se distribuye de forma distinta según el país?

* Sobre el soporte
6. Los tickets de soporte se concentran en algún plan en particular?

* Sobre la estructura de los datos (PCA)
7. Las variables numéricas estandarizadas se pueden resumir en pocas componentes principales sin perder demasiada varianza?
8. Al proyectar a los usuarios sobre las dos primeras componentes y colorear por plan (o país), se distinguen perfiles de usuario?
